# Results 1 — Panoramic view across 100,526 complex trait GWAS

Every number claimed in this Results subsection, and the per-year discovery curves behind
Figure 1b–d, Extended Data Figures 3 and 10.

Numbers are collected in `numbers` and written to `results/panoramic.json`, which
`tools/check_numbers.py` compares against the manuscript.

In [1]:
import pandas as pd
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from pyspark.sql import functions as f

from manuscript_methods import discovery, paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})
numbers = {}

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/19 00:59:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Studies, publications and ontology terms

In [2]:
si = StudyIndex.from_parquet(session, paper.release("study")).df.cache()
gwas = si.filter(f.col("studyType") == "gwas").cache()

numbers["R1.01"] = gwas.count()
numbers["R1.04"] = len(paper.THERAPEUTIC_AREAS)
print("GWAS studies:", numbers["R1.01"], "| therapeutic areas:", numbers["R1.04"])

26/08/19 00:59:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


GWAS studies: 100526 | therapeutic areas: 23


## Ancestry composition of the study set

A study counts as having more than 10% non-European participants when non-Finnish Europeans
do not reach 90% of its LD population structure. Studies with no LD population structure are
outside this denominator.

In [3]:
populations = (
    gwas.select("studyId", "ldPopulationStructure", "publicationDate")
    .withColumn("ldPop", f.explode("ldPopulationStructure"))
    .withColumn("year", f.year(f.to_date("publicationDate", "yyyy-MM-dd")))
    .cache()
)
predominantly_european = (f.col("ldPop.ldPopulation") == "nfe") & (f.col("ldPop.relativeSampleSize") >= 0.9)


def non_european_share(rows):
    """Percentage of studies where non-Finnish Europeans do not reach 90%."""
    total = rows.select("studyId").distinct().count()
    european = rows.filter(predominantly_european).select("studyId").distinct().count()
    return round(100 * (1 - european / total), 1)


numbers["R1.05"] = non_european_share(populations.filter(f.col("year") <= 2017))
numbers["R1.06"] = non_european_share(populations)
print({k: numbers[k] for k in ["R1.05", "R1.06"]})

{'R1.05': 23.2, 'R1.06': 35.1}


## Credible sets

In [4]:
cs = StudyLocus.from_parquet(session, paper.release("credible_set")).df.cache()
gwas_cs = cs.filter(f.col("studyType") == "gwas").cache()

numbers["R1.07"] = gwas_cs.count()
numbers["R1.08"] = gwas_cs.select("studyId").distinct().count()

# Publications and ontology terms are counted over the studies that produced a credible set.
studies_with_cs = gwas.join(gwas_cs.select("studyId").distinct(), "studyId", "inner").cache()
numbers["R1.02"] = studies_with_cs.select("pubmedId").distinct().count()
numbers["R1.03"] = (
    gwas.join(
        session.spark.read.parquet(paper.derived("qualifying_gwas_studies"))
        .select("studyId")
        .union(session.spark.read.parquet(paper.derived("qualifying_measurement_studies")).select("studyId")),
        "studyId",
        "inner",
    )
    .select(f.explode("diseaseIds").alias("id"))
    .distinct()
    .count()
)
print({k: numbers[k] for k in ["R1.02", "R1.03"]})
numbers["R1.15"] = cs.filter(f.col("studyType") != "gwas").count()
numbers["R1.16"] = (
    si.join(cs.filter(f.col("studyType") != "gwas").select("studyId").distinct(), "studyId", "inner")
    .select("biosampleId")
    .distinct()
    .count()
)
print({k: numbers[k] for k in ["R1.07", "R1.08", "R1.15", "R1.16"]})

{'R1.02': 4250, 'R1.03': 9147}


{'R1.07': 789453, 'R1.08': 39282, 'R1.15': 2044305, 'R1.16': 98}


## Qualifying credible sets and the variants in them

In [5]:
disease_cs = session.spark.read.parquet(paper.derived("qualifying_credible_sets")).select("studyLocusId")
measurement_cs = session.spark.read.parquet(paper.derived("qualifying_measurement_credible_sets")).select(
    "studyLocusId"
)

numbers["R1.10"] = disease_cs.count()
numbers["R1.11"] = measurement_cs.count()
numbers["R1.09"] = numbers["R1.10"] + numbers["R1.11"]

qualifying = cs.join(disease_cs.union(measurement_cs).distinct(), "studyLocusId", "inner").cache()
locus_variants = qualifying.select(f.explode("locus").alias("l"))

numbers["R1.13"] = qualifying.select("variantId").distinct().count()
numbers["R1.12"] = locus_variants.select("l.variantId").distinct().count()
numbers["R1.14"] = (
    locus_variants.filter(f.col("l.posteriorProbability") >= 0.9).select("l.variantId").distinct().count()
)
print({k: numbers[k] for k in ["R1.09", "R1.10", "R1.11", "R1.12", "R1.13", "R1.14"]})

{'R1.09': 520975, 'R1.10': 70618, 'R1.11': 450357, 'R1.12': 2024916, 'R1.13': 211597, 'R1.14': 49772}


## Prioritised genes, and the traits they cover

In [6]:
diseases = session.spark.read.parquet(paper.derived("prioritised_genes_diseases")).cache()
measurements = session.spark.read.parquet(paper.derived("prioritised_genes_measurements")).cache()
target = session.spark.read.parquet(paper.release("target"))

numbers["R1.17"] = diseases.count() + measurements.count()
numbers["R1.18"] = diseases.select("geneId", f.explode("diseaseIds").alias("traitId")).distinct().count()
numbers["R1.19"] = measurements.select("geneId", f.explode("diseaseIds").alias("traitId")).distinct().count()

disease_genes = diseases.select("geneId").distinct()
measurement_genes = measurements.select("geneId").distinct()
numbers["R1.21"] = disease_genes.count()
numbers["R1.22"] = measurement_genes.count()
numbers["R1.20"] = disease_genes.union(measurement_genes).distinct().count()
numbers["R1.23"] = diseases.select(f.explode("diseaseIds").alias("traitId")).distinct().count()
numbers["R1.24"] = measurements.select(f.explode("diseaseIds").alias("traitId")).distinct().count()

# The published percentage is taken against the "All genes" reference set, the protein-coding
# universe used throughout the gene-set analysis.
gene_sets = session.spark.read.parquet(paper.derived("gene_sets"))
protein_coding = gene_sets.filter(f.col("geneSet") == "All genes").select("geneId").distinct().count()
numbers["R1.25"] = round(100 * numbers["R1.20"] / protein_coding, 1)
print({k: numbers[k] for k in ["R1.17", "R1.18", "R1.19", "R1.20", "R1.21", "R1.22", "R1.23", "R1.24", "R1.25"]})
print("protein-coding genes in the release:", protein_coding)

{'R1.17': 523409, 'R1.18': 36858, 'R1.19': 150360, 'R1.20': 15641, 'R1.21': 8285, 'R1.22': 15160, 'R1.23': 1394, 'R1.24': 3412, 'R1.25': 77.9}
protein-coding genes in the release: 20083


## Discovery over time

Cumulative discovery for the nested ancestry tiers of Figure 1c: EUR common, then non-EUR
common, then mixed common, then rare variants of any ancestry. Each tier's `layer` is what
it adds over the tier below.

In [7]:
disease_rows = diseases.select("geneId", "diseaseIds", "year", "ancestryClass", "freqClass").toPandas()
disease_pairs = discovery.explode_pairs(disease_rows)

genes_nested = discovery.nested_tiers(disease_rows, ["geneId"], "disease genes")
pairs_nested = discovery.nested_tiers(disease_pairs, ["geneId", "traitId"], "gene-disease pairs")
fig1c = pd.concat([genes_nested, pairs_nested], ignore_index=True)
fig1c.to_csv(paper.derived("fig1c_cumulative_discovery.csv"), index=False)

fig1c.pivot_table(index="year", columns=["metric", "tier_index"], values="cumulative").tail(6)

metric     disease genes                         gene-disease pairs           \
tier_index             1       2       3       4                  1        2   
year                                                                           
2019              2722.0  2986.0  3124.0  3145.0             7051.0   7642.0   
2020              3101.0  3499.0  3755.0  3781.0             8199.0   9085.0   
2021              3919.0  4362.0  4747.0  4789.0            11005.0  12177.0   
2022              4513.0  4992.0  5437.0  5494.0            13202.0  14543.0   
2023              5038.0  5557.0  6056.0  6119.0            15332.0  16913.0   
2024              5552.0  7381.0  8014.0  8129.0            18521.0  30806.0   

metric                        
tier_index        3        4  
year                          
2019         8003.0   8040.0  
2020         9688.0   9738.0  
2021        13429.0  13560.0  
2022        16100.0  16275.0  
2023        18670.0  18878.0  
2024        34905.0  35535.0

In [8]:
def tier_totals(nested, metric):
    """Final-year cumulative total per tier, and what each tier adds."""
    end = nested[(nested["metric"] == metric) & (nested["year"] == discovery.MAX_YEAR)].sort_values("tier_index")
    out = end[["tier", "tier_index", "cumulative"]].copy()
    out["increment"] = out["cumulative"].diff().fillna(out["cumulative"]).astype(int)
    return out.reset_index(drop=True)


gene_tiers = tier_totals(fig1c, "disease genes")
pair_tiers = tier_totals(fig1c, "gene-disease pairs")
print(gene_tiers.to_string(index=False))
print()
print(pair_tiers.to_string(index=False))

                          tier  tier_index  cumulative  increment
                  EUR (common)           1        5552       5552
        EUR + non-EUR (common)           2        7381       1829
EUR + non-EUR + mixed (common)           3        8014        633
              all (incl. rare)           4        8129        115

                          tier  tier_index  cumulative  increment
                  EUR (common)           1       18521      18521
        EUR + non-EUR (common)           2       30806      12285
EUR + non-EUR + mixed (common)           3       34905       4099
              all (incl. rare)           4       35535        630


In [9]:
# Tier 3 is all common variants of any ancestry; tier 1 is EUR common only.
for prefix, tiers in [("R1.2", gene_tiers), ("R1.3", pair_tiers)]:
    common_total = int(tiers.loc[tiers["tier_index"] == 3, "cumulative"].iloc[0])
    eur_common = int(tiers.loc[tiers["tier_index"] == 1, "cumulative"].iloc[0])
    non_eur = int(tiers.loc[tiers["tier_index"] == 2, "increment"].iloc[0])
    mixed = int(tiers.loc[tiers["tier_index"] == 3, "increment"].iloc[0])
    all_total = int(tiers.loc[tiers["tier_index"] == 4, "cumulative"].iloc[0])
    if prefix == "R1.2":
        # The genes claim is quoted against the rare-inclusive total, the pairs claim against
        # the common-only total; both are reported here as the manuscript states them.
        numbers["R1.26"], numbers["R1.27"] = common_total - eur_common, all_total
        numbers["R1.28"], numbers["R1.29"] = non_eur, mixed
    else:
        numbers["R1.30"], numbers["R1.31"] = common_total - eur_common, common_total
        numbers["R1.32"], numbers["R1.33"] = non_eur, mixed
    print(
        f"{prefix}: EUR common {eur_common}, +non-EUR {non_eur}, +mixed {mixed}, all common {common_total}, incl. rare {all_total}"
    )

R1.2: EUR common 5552, +non-EUR 1829, +mixed 633, all common 8014, incl. rare 8129
R1.3: EUR common 18521, +non-EUR 12285, +mixed 4099, all common 34905, incl. rare 35535


## Extended Data Figures 3 and 10

The same tiers for measurements (Extended Data Fig. 3), and the rare-variant share of
cumulative discovery (Extended Data Fig. 10). The rare share is the reachability difference
between the top two tiers: entities that no common-variant study of any ancestry reaches.

In [10]:
measurement_rows = measurements.select(
    "geneId", "studyId", "diseaseIds", "year", "ancestryClass", "freqClass"
).toPandas()

ed3 = pd.concat(
    [
        discovery.nested_tiers(measurement_rows, ["geneId"], "measurement genes"),
        discovery.nested_tiers(measurement_rows, ["geneId", "studyId"], "gene-measurement pairs (studyId)"),
    ],
    ignore_index=True,
)
ed3.to_csv(paper.derived("ed3_cumulative_discovery.csv"), index=False)
ed3.pivot_table(index="year", columns=["metric", "tier_index"], values="cumulative").tail(4)

metric     gene-measurement pairs (studyId)                                \
tier_index                                1         2         3         4   
year                                                                        
2021                               111674.0  118596.0  139876.0  143353.0   
2022                               137368.0  146312.0  170471.0  174597.0   
2023                               160411.0  175133.0  201605.0  206202.0   
2024                               251690.0  282243.0  350431.0  358449.0   

metric     measurement genes                             
tier_index                 1        2        3        4  
year                                                     
2021                 11398.0  11732.0  12149.0  12280.0  
2022                 12075.0  12479.0  12875.0  13005.0  
2023                 12430.0  12954.0  13320.0  13436.0  
2024                 13914.0  14637.0  14930.0  15061.0

In [11]:
rare = pd.concat(
    [
        discovery.rare_share(fig1c, "disease genes").assign(panel="a"),
        discovery.rare_share(fig1c, "gene-disease pairs").assign(panel="b"),
    ],
    ignore_index=True,
).rename(columns={"total": "all_variants_cumulative", "rare": "rare_cumulative", "rare_pct": "rare_share_pct"})
rare.to_csv(paper.derived("rare_discovery_over_time.csv"), index=False)
rare[rare["year"] >= 2020].round(2)

,metric,year,all_variants_cumulative,rare_cumulative,rare_share_pct,panel
14,disease genes,2020,3781.0,26.0,0.69,a
15,disease genes,2021,4789.0,42.0,0.88,a
16,disease genes,2022,5494.0,57.0,1.04,a
17,disease genes,2023,6119.0,63.0,1.03,a
18,disease genes,2024,8129.0,115.0,1.41,a
33,gene-disease pairs,2020,9738.0,50.0,0.51,b
34,gene-disease pairs,2021,13560.0,131.0,0.97,b
35,gene-disease pairs,2022,16275.0,175.0,1.08,b
36,gene-disease pairs,2023,18878.0,208.0,1.10,b
37,gene-disease pairs,2024,35535.0,630.0,1.77,b


## Pleiotropy of disease-associated genes

The gene-level table restated: how many disease genes carry more than one disease, and more
than one therapeutic area.

In [12]:
gene_table = session.spark.read.parquet(paper.derived("gene_table")).toPandas()
numbers["R1.37"] = int((gene_table["uniqueDiseases"] >= 2).sum())
numbers["R1.38"] = int((gene_table["uniqueTherapeuticAreas"] > 1).sum())
print({k: numbers[k] for k in ["R1.37", "R1.38"]})

{'R1.37': 5314, 'R1.38': 4743}


## Figure 1b inputs — sample size and effect size over time

In [13]:
for name, subset in [("qd_sl_eff", "qualifying_credible_sets"), ("qm_sl_eff", "qualifying_measurement_credible_sets")]:
    table = (
        session.spark.read.parquet(paper.derived(subset))
        .join(
            session.spark.read.parquet(paper.derived("study_annotation")).select("studyId", "year"), "studyId", "inner"
        )
        .select(
            "studyId",
            "studyLocusId",
            "year",
            f.col("studyStatistics.nSamples").alias("nSamples"),
            f.abs(f.col("rescaledStatistics.absEstimatedBeta")).alias("absBeta"),
            f.col("majorLdPopulationMaf.value").alias("maf"),
        )
        .toPandas()
    )
    table.to_csv(paper.derived(f"{name}.csv"), index=False)
    print(name, table.shape)

qd_sl_eff (70618, 6)


qm_sl_eff (450357, 6)


## Numbers

In [14]:
print(paper.save_results("panoramic", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/panoramic.json


,computed
R1.01,100526.0
R1.04,23.0
R1.05,23.2
R1.06,35.1
R1.07,789453.0
R1.08,39282.0
R1.02,4250.0
R1.03,9147.0
R1.15,2044305.0
R1.16,98.0
